In [ ]:
import pandas as pd
from pandas import DataFrame
import spacy
from spacy import displacy
from spacy.tokens import DocBin
import json
from datetime import datetime
from tqdm import tqdm
import re
import pickle
import spacy_transformers # for en_core_web_trf

EQUAL WEIGHTS, NOTHING DONE ABOUT ASSIGNING DIFFERENT WEIGHTS TO GOLDEN, SILVER, BRONZE DATASETS.

Look into example.weight in spaCy, I didn't manage to do that.

Otherwise do resampling with replacement, if you wanna run with diffferent weight assigned to each dataset.

I did my own preprocessing, it is probably missing something. Unfortunately, I didn't have time to do it like in Utils

In [ ]:

# load the preprocessed data
with open("train_data.pkl", "rb") as f:
    train_data = pickle.load(f)

with open("dev_data.pkl", "rb") as f:
    dev_data = pickle.load(f)

In [ ]:
# create a blank model
nlp = spacy.load('en_core_web_trf')

In [ ]:
def create_training(train_data):
    db = DocBin()
    entities_skipped = 0
    start_end_idx = []
    for text, annot in tqdm(train_data):
        doc = nlp.make_doc(text)
        ents = []

        # create span objects
        for start, end, label in annot["entities"]:
            # adding 
            #if start > end:
                #start, end = end, start
            #span = doc.char_span(start, end , label = label, alignment_mode = "strict") 
            
            print(f"Start: {start}, End: {end + 1}")
            print(f"Substring: '{doc.text[start:end + 1]}'")
            print(f"Tokens around this area:")
            #for token in doc:
                #if token.idx >= start - 5 and token.idx <= end + 5:
                    #print(f"  Token: '{token.text}', idx: {token.idx}, end: {token.idx + len(token.text)}")
            
            # end + 1 is used because in SpaCy the last number in the interval is not included in the index
            span = doc.char_span(start, end + 1, label=label, alignment_mode = "expand")
            print(f"Span: {span}")
            print("---")

            # skip if the character indices do not map to a valid span
            if span is None:
                print("Skipping entity.")
                entities_skipped += 1
                start_end_idx.append({start, end})
            else:
                ents.append(span)
                # handle erroneous entity annotations by removing them
                try:
                    doc.ents = ents
                except:
                    print("BAD SPAN:", span, "\n")
                    ents.pop()
        doc.ents = ents

        # pack Doc objects into DocBin
        db.add(doc)
    return db


train_data_doc = create_training(train_data)
data_dev_doc = create_training(dev_data)

In [ ]:
# exporting the data
train_data_doc.to_disk("train_data.spacy")
data_dev_doc.to_disk("data_dev.spacy")

Open your CLI and cd over into the directory of base_config.cfg
Simply copy/paste this command: (might be that you don't need the exclamation mark in the beginnings')
    
!python -m spacy init fill-config base_config.cfg config.cfg

A config.cfg file will appear in your working directory.

Next, run the following to begin training:
    
!python -m spacy train config.cfg --output ./output

After training is complete, the resulting model will appear in a new folder called output.

In [ ]:
# MODEL RESULTS (for fun)

'''
I chose a pubmed article, that is not present in the datasets

pmid =  32526057


articles_title_abstract.loc[articles_title_abstract['pmid'] == 32526057]
articles_dev.loc[articles_dev['pmid'] == 32526057]
'''

model_test = "Amyotrophic lateral sclerosis (ALS) is a neurodegenerative disorder affecting primarily \
    the motor system, but in which extra-motor manifestations are increasingly recognized. The loss of upper \
    and lower motor neurons in the motor cortex, the brain stem nuclei and the anterior horn of the spinal \
    cord gives rise to progressive muscle weakness and wasting. ALS often has a focal onset but \
    subsequently spreads to different body regions, where failure of respiratory muscles typically \
    limits survival to 2-5 years after disease onset. In up to 50% of cases, there are extra-motor \
    manifestations such as changes in behaviour, executive dysfunction and language problems. \
    In 10%-15% of patients, these problems are severe enough to meet the clinical criteria of frontotemporal \
    dementia (FTD). In 10% of ALS patients, the family history suggests an autosomal dominant inheritance pattern. \
    The remaining 90% have no affected family members and are classified as sporadic ALS. The causes of ALS appear \
    to be heterogeneous and are only partially understood. To date, more than 20 genes have been associated with ALS. \
    The most common genetic cause is a hexanucleotide repeat expansion in the C9orf72 gene, responsible for 30%-50% of \
    familial ALS and 7% of sporadic ALS. These expansions are also a frequent cause of frontotemporal dementia, \
    emphasizing the molecular overlap between ALS and FTD. To this day there is no cure or effective treatment for ALS and the \
    cornerstone of treatment remains multidisciplinary care, including nutritional and respiratory support and symptom management. \
    In this review, different aspects of ALS are discussed, including epidemiology, aetiology, pathogenesis, clinical features, differential \
    diagnosis, investigations, treatment and future prospects."


# load the trained model
nlp_output = spacy.load("output/model-best")

# pass our test instance into the trained pipeline
doc = nlp_output(model_test)


# visualize the identified entities
displacy.render(doc, style = "ent")

visualizer = displacy.render(doc, style = "ent")

with open("entities.html", "w") as f:
    f.write(visualizer)

